# 11 Ignore Index


Chạy sau notebook 14. IGNORE positions không đóng góp vào cross-entropy loss.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLP/lab09'
import os, json, random, numpy as np
os.makedirs(DRIVE_DIR, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
!pip -q install datasets==3.6.0 transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import torch.nn.functional as F
ds=load_dataset('eriktks/conll2003', trust_remote_code=True)
label_names=ds['test'].features['ner_tags'].feature.names
model_dir=f'{DRIVE_DIR}/bert-base-cased-conll2003'
tokenizer=AutoTokenizer.from_pretrained(model_dir, use_fast=True)
model=AutoModelForTokenClassification.from_pretrained(model_dir)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device).eval()
row=ds['test'][0]
enc=tokenizer(row['tokens'],is_split_into_words=True,truncation=True,max_length=256)
word_ids=enc.word_ids(); aligned=[]; previous=None
for wid in word_ids:
    aligned.append(-100 if wid is None or wid==previous else row['ner_tags'][wid])
    previous=wid
with torch.no_grad():
    batch={k:torch.tensor([v]).to(device) for k,v in enc.items()}
    logits=model(**batch).logits[0].cpu()
labels=torch.tensor(aligned)
valid=labels.ne(-100)
loss_ignore=F.cross_entropy(logits,labels,ignore_index=-100)
loss_valid=F.cross_entropy(logits[valid],labels[valid])
print('tokens:',tokenizer.convert_ids_to_tokens(enc['input_ids']))
print('aligned_labels:',aligned)
print('ignored_positions:',int((~valid).sum()))
print('positions_used_in_loss:',int(valid.sum()))
print('loss_with_ignore_index:',float(loss_ignore))
print('loss_on_valid_positions:',float(loss_valid))
assert torch.allclose(loss_ignore,loss_valid)
